# 3  - Sailkapena

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1 - Datuak lortu

* **Breast Cancer Wisconsin**: *Tumore bat gaiztoa (malignant) ala onbera (benign) den iragartzea, zelulen neurri fisikoetan oinarrituz.*
   * Zuzenean *Scikit-learn*-etik kargatu daiteke `sklearn.datasets` modulua erabiliz
   * Eremuak dituen hiztegi gisako objektu bat:
      * `o.data` edo `o['data']`: aldagai iragarleak (*Numpy*-ko matrizea)
      * `o.target` edo `o['target']`: helburu-aldagaia (*Numpy*-ko arraya)
      * `o.feature_names` edo `o['feature_names']`: aldagai iragarleen izenak

In [ ]:
from sklearn import datasets

cancer = datasets.load_breast_cancer()
print(type(cancer))
#cancer
#cancer.data
#cancer['data']

Datuak DataFrame batetan jar ditzakegu:

In [ ]:
# DataFrame-a sortu aldagai iragarleekin
df = pd.DataFrame(data=cancer.data, columns=cancer.feature_names)

# Helburu-aldagaia gehitu (0: gaiztoa / malignant, 1: onbera / benign)
df['target'] = cancer.target

df

## 2 - Datuak aztertu

In [ ]:
df.info()

In [ ]:
df.hist(bins=50, figsize=(15,20));

## 3 - Train/Test banaketa egin

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f'{df.shape = }')
print(f'{type(train) = }  {train.shape = }')
print(f'{type(test) = }  {test.shape = }')
#train
#test

## 4 - Aldagai iragarleak eta helburu-aldagaia ezberdindu 

In [ ]:
X_train = train.drop(columns=['target'])
y_train = train['target']
X_test = test.drop(columns=['target'])
y_test = test['target']
print(f'{type(X_train) = }')
print(f'{type(y_train) = }')
#X_train
#y_train
#X_test
#y_test

## 5 - Ereduak sortu, doitu eta ebaluatu

Atal honetan `ml_course` moduluak eskaintzen digun sailkapen eredu bat erabiliko dugu:
* `ACMEClassifier`: *A simple classification model to distinguish between different categories*

Modulua instalatu eta bi ereduak inportatu:

In [ ]:
!pip install git+https://github.com/mpenagar/ml_course.git
#import ml_course
#import importlib
#importlib.reload(ml_course)
from ml_course import ACMEClassifier

In [ ]:
?ACMEClassifier

In [ ]:
model = ACMEClassifier()
#model = ACMEClassifier(regularization_strength=10, max_iterations=10000, random_state=42)
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
#y_train_predict
#y_test_predict

### Zehaztasuna (*Accuracy*)

Sailkapen-eredu batek zuzen asmatutako iragarpenen proportzioa da. Ereduak aztertutako kasu guztietatik zenbatetan asmatu duen adierazten du ehuneko edo proportzio gisa:

$$\text{Accuracy} = \frac{\text{Iragarpen Zuzenak}}{\text{Iragarpen Guztiak}}$$

In [ ]:
guztiak = y_test.size
zuzenak = (y_test_predict == y_test).sum()
zehaztasuna = zuzenak / guztiak *100
print(f'{guztiak = }  {zuzenak = }  {zehaztasuna = :.2f}%')

*Scikit-learn*-ek zehaztasuna kalkulatzen duen `accuracy_score` eskaintzen digu:

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_train = accuracy_score(y_train, y_train_predict)
accuracy_test = accuracy_score(y_test, y_test_predict)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

*Scikit-learn*-en ereduek (eta antzekoak ere, gure kasuan `ml_course` moduluko `ACMEClassifier` ereduak) zehaztasuna kalkulatzen duen `score` metodoa eskaintzen dute:

```python
model = SomeClassificationModel(...)
model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)
```

In [ ]:
accuracy_train = model.score(X_train, y_train)
accuracy_test = model.score(X_test, y_test)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

## 6 - Ereduaren hiperparametroak doitzen

Aurreko gaiean ikusi genuen bezala, *Scikit-learn*-eko `GridSearchCV` erabil dezakegu:

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {
    'regularization_strength' : [0.01,0.1,1,10,100]
}

grid_search = GridSearchCV(
    estimator=ACMEClassifier(max_iterations=10000, random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

In [ ]:
print(f'{grid_search.best_params_ = } , {grid_search.best_score_ = }')

`GridSearchCV` exekutatzean agertzen diren *warning* guztiak aurrerago aztertuko ditugu. Oraingoz lortutako ereduaren errendimendua neurtuko dugu:

In [ ]:
model = grid_search.best_estimator_
accuracy_train = model.score(X_train, y_train)
accuracy_test = model.score(X_test, y_test)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

## 7 - Konbergentzia arazoak

`GridSearchCV` exekutatzean eredua askotan doitzen denez (denera 25 aldiz), eskuz jarri dugun `max_iter=10000` ez da nahikoa kasu guztietarako eta konbergentzia arazoak gertatzen dira. *Warning*-ak berak bi irtenbide gomendatzen dizkigu:

1. `max_iter` handitzea
   * Dagoeneko egin dugu, defektuzko balioa 100 zen.
   * Gehiago handitzen badugugu konbergentzira iritsi gintezke, baina seguraski gehiegi luzatuko dugu prozesua.
1. Datuak eskalatzea
   * Adibidez aurreko gaiko estandarizazioa erabiliz (`sklearn.preprocessing.StandardScaler`)
   * Arazoa datuen magnitudeak eragin badu, eskalatzeak konbergentziaren arazoa baztertuko du eta doikuntza azkarragoa izango da 
  
`StandardScaler`-en `transform()` metodoak *Numpy*-ko matrize bat bueltatuko badu ere, aldagai iragarle guztiak batera estandarizatu daitezke:

```python
scaler = StandardScaler()
scaler.fit(df)
df[:] = scaler.transform(df) # Dataframe baten zutabe guztiei Numpy-ko matrize bat esleitu
# beste aukera bat:
# df.iloc[:, :] = scaler.transform(df)
```

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler() 
scaler.fit(X_train)
print(f'{type(scaler.transform(X_train))=}  {scaler.transform(X_train).shape=}')
X_train[:] = scaler.transform(X_train)
X_test[:] = scaler.transform(X_test)
X_train.hist(bins=50, figsize=(15,20));

Saia gaitezen orain defektuzto `max_iter=100` erabiltzen (defektuzkoa denez, ez dugu jarriko):

In [ ]:
grid = {
    'regularization_strength' : [0.01,0.1,1,10,100]
}
grid_search = GridSearchCV(
    estimator=ACMEClassifier(random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
print(f'{grid_search.best_params_ = } , {grid_search.best_score_ = }')

Aurrekoa baina dexente bizkorrakoa, *warning*-ak desagertu dira eta bide batez errendimendua (5-fold CV-koa) ere hobetu dugu (96,26% &rarr; 97.14%)

Nahi izanez gero, `regularization_strength` hiperparametroaren balioa hobeto doitzen saia gintezke (`10` balioaren inguruan):

In [ ]:
grid = {
    'regularization_strength' : [2.5,5,10,20,40]
}
grid_search = GridSearchCV(
    estimator=ACMEClassifier(max_iterations=100, random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
print(f'{grid_search.best_params_ = } , {grid_search.best_score_ = }')

97.14% &rarr; 97.36% , 5-en inguruan jarraitu dezakedu:

In [ ]:
grid = {
    'regularization_strength' : [3,4,5,6,7]
}
grid_search = GridSearchCV(
    estimator=ACMEClassifier(max_iterations=100, random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
print(f'{grid_search.best_params_ = } , {grid_search.best_score_ = }')

Hobekuntzarik ez dagoenez, bilaketa amaitutzat eman eta ereduaren errendimendua neurtu dezakegu test azpimultzoan:

In [ ]:
model = grid_search.best_estimator_
accuracy_train = model.score(X_train, y_train)
accuracy_test = model.score(X_test, y_test)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

| System          | $\mathbf{Accuracy_{train}}$ | $\mathbf{Accuracy_{test}}$ |
|-----------------|---------------:|---------------:|
| **ACMEClassifier** |       98.68% |      98.25% |

## 8 - *Baseline* erreferentzia-eredua

Aurreko gaiean ikusi genuen bezala, ondo legoke erreferentziazko eredu inozo bat izatea. Aukerak:

1. Auzazki (`0`/`1`) erantzuten duen eredua
1. Beti berdina erantzuten duen eredua
   * `0` edo `1`? &rarr; Entrenamenduan gehienetan agertzen dena

### *Baseline* - Auzazko eredua

In [ ]:
np.random.seed(42)
y_train_predict = np.random.randint(0, 2, size=y_train.size)
y_test_predict = np.random.randint(0, 2, size=y_test.size)
accuracy_train = accuracy_score(y_train, y_train_predict)
accuracy_test = accuracy_score(y_test, y_test_predict)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

### *Baseline* - Moda eredua

Klase bakoitza zenbatetan agertzen den aztertu dezakegu:

In [ ]:
print(f'{(y_train==0).sum() = :d}')
print(f'{(y_train==1).sum() = :d}')

*Pandas*-eko serieek `moda()` metodoa definitua dute. Moda elementu bat baina gehiago izan daitekeenez, modaren emaitza beste Series bat izango da:

In [ ]:
s = pd.Series(['a','b','c','a','b','c','a','c'])
print(f'{type(s.mode()) = }')
s.mode()

Modetako elementuren bat lortzeko, beti lehenengoarekin gera gintezke, `s.mode()[0]`. Gure kasuan:

In [ ]:
y_mode = y_train.mode()[0]
print(f'{y_mode = :d}')
y_train_predict = np.full(y_train.shape, y_mode)
y_test_predict = np.full(y_test.shape, y_mode)
accuracy_train = accuracy_score(y_train, y_train_predict)
accuracy_test = accuracy_score(y_test, y_test_predict)
print(f"Ereduaren zehaztasuna - train: {accuracy_train*100:.2f}%  test: {accuracy_test*100:.2f}%")

Laburbilduz:

| System          | $\mathbf{Accuracy_{train}}$ | $\mathbf{Accuracy_{test}}$ |
|-----------------|---------------:|---------------:|
| **Baseline-rand** |        49.67% |      50.88% |
| **Baseline-mode** |        62.86% |      62.28% |
| **ACMEClassifier** |       98.68% |      98.25% |


### Hausnarketa

Demagun gure minbiziaren inguruko datubasean, laginen % 99a minbizi onberak direla. Bi erreferentzia ereduen errendimendua ondokoa litzateke:

| System          | $\mathbf{Accuracy}$ |
|-----------------|---------------:|
| **Baseline-rand** |        $\approx$ 50% |
| **Baseline-mode** |        $\approx$ 99% |

Argi ikus daitekeenez, erreferentziazko eredu baten errendimentua % 99koa bada, errendimendu metrika horrek ez digu askorako balio izango. Izan ere, errendimendua neurtzeko erabiltzen dugun azpimultzoa bitan zatituko bagenu, alde batetik minbizi onberak (`target = 1`) eta bestetik gaiztoak (`target = 0`), gauzek beste itxura bat izango lukete: 

| System          | $\mathbf{Accuracy_{target=1}}$ | $\mathbf{Accuracy_{target=0}}$ |
|-----------------|---------------:|---------------:|
| **Baseline-rand** |        $\approx$ 50% |      $\approx$ 50% |
| **Baseline-mode** |        100% |      0% |

Klase bakoitzaren zehaztasunaren batazbestekoa kalkulatu izan bagenu (ondoren ikusiko dugunez, honi *Macro-Accuracy* deritzo):

| System          | $\mathbf{Macro-Accuracy}$ |
|-----------------|---------------:|
| **Baseline-rand** |        $\approx$ 50% |
| **Baseline-mode** |        50% |


## 9 - Nahasmen matrizea

Aurreko ataleko zehaztasun errendimendu metrika (baita *Macro-Accuracy*a ere). Sailkapen ereduak bere iragarpenetan egiten dituen erroreetan oinarritua dago. **Nahasmen-matrizea** datuen benetako klaseen eta sailkapen-ereduak iragarritako klaseen zenbatekoak alderatzen dituen taula da. Sailkapen bitar batean, lau gelaxka nagusi ditu:

| ${}_{Benetakoa}\mkern-6mu\diagdown\mkern-6mu{}^{Iragarpena}$ | Negatiboa (0) | Positiboa (1) |
|:-----------------:|:---------------:|:---------------:|
| **Negatiboa (0)** |  Egiazko Negatiboak (**TN**) |  Positibo Faltsuak (**FP**) |
| **Positiboa (1)** |  Negatibo Faltsuak (**FN**) | Egiazko Positiboak (**TP**) |

Beraz, ereduaren asmatze guztiak diagonalean agertzen dira, erroreak ordea diagonaletik kanpo:
* Egiazko Negatiboak (**TN**, *True Negative*): Ondo sailkatutako negatibo kopurua
* Egiazko Positiboak (**TP**, *True Positive*): Ondo sailkatutako positibo kopurua
* Positibo Faltsuak (**FP**, *False Positive*): Sailkapen erroreak, negatiboa zena positibo bezala iragarri diren kopurua
* Negatibo Faltsuak (**FN**, *False Negative*): Sailkapen erroreak, positibo zena negatiboa bezala iragarri diren kopurua

Nahasmen matrizean eredu baten asmatze eta errore mota guztiak agertzen badira ere, ezin dugu zuzenean errendimendu metrika gis bezala erabili, bi halako matrize konpara ezinak direlako. Demagun bi eredu ditugula ondoko nahasmen matrizea dutena:

| ${}_{Benetakoa}\mkern-6mu\diagdown\mkern-6mu{}^{Iragarpena}$ | Negatiboa (0) | Positiboa (1) |
|:-----------------:|:---------------:|:---------------:|
| **Negatiboa (0)** |  123 |  17 |
| **Positiboa (1)** |  54 | 234 |

| ${}_{Benetakoa}\mkern-6mu\diagdown\mkern-6mu{}^{Iragarpena}$ | Negatiboa (0) | Positiboa (1) |
|:-----------------:|:---------------:|:---------------:|
| **Negatiboa (0)** |  113 |  27 |
| **Positiboa (1)** |  44 | 244 |

Bi ereduetatik zein da hobea?

Sailkapen errendimendu metrika gehienak nahasmen matrizearen balioetatik abiatzen dira, <u>konparagarria izango den errendimendu metrika eskalar</u> bat lortzeko.

### Zehaztasuna (*Accuracy*)

Zuzen asmatutako iragarpenen proportzioa adierazten du:

$$\text{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$


### Doitasuna (*Precision*)

Positibo gisa iragarritakoetatik, zuzen asmatutako iragarpenen proportzioa adierazten du:

$$\text{Precision} = \frac{TP}{TP+FP}$$


### Estaldura edo Sentikortasuna (*Recall* or *Sensitivity*)

Benetako positiboetatik, zuzen asmatutako iragarpenen proportzioa adierazten du:

$$\text{Precision} = \frac{TP}{TP+FN}$$

### Hausnarketa

Doitasuna (*Precision*) eta Estaldura (*Recall*) ezin dira errendimendu-metrika isolatu gisa erabili. Suposa ditzagun ondoko bi ereduak:

* **Beti `1` iragartzen duen eredua**. Sekula `0` iragarriko ez duenez, $FN=0$ eta $TP>0$, beraz $\text{Recall}=1$
* **Guztiz seguru dagoenean soilik `1` iragartzen duen eredua**. Eredu honen kasuan $FP \approx 0$ eta $TP>0$, beraz $\text{Precision} \approx 1$

Argi gelditzen da metrika hauek, isolatuki, ezin direla izan helburuak izan, nahiko erraz lor daitekeelako bietako bat maximizatzen duen eredua. Biak batera maximizatzea beste kontu bat litzateke...

### F1-score

Doitasunaren eta estalduraren arteko batez besteko harmonikoa da:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Nolabait, aurreko bi metriken arteko oreka bilatzen duen metrika bat da.

### Egiazko Positiboen Tasa (TPR, *True Positive Rate*)

Estalduraren (*Recall*) berdina da, benetan positiboak diren kasu guztietatik ereduak asmatu dituen positiboen tasa neurtzen du

$$\text{TPR} = \frac{TP}{TP+FN} = \text{Recall}$$

### Positibo Faltsuen Tasa (FPR, *False Positive Rate*)

Benetan negatiboak diren kasu guztietatik ereduak positibo gisa oker iragarri dituenen tasa da.

$$\text{FPR} = \frac{FP}{FP+TN}$$

**OHARRA**: FPR errore metrika bat da, ez errendimendu metrika. FPR txikiagoak eredu hobea adierazten du.

### Hausnarketa

Doitasuna (*Precision*) eta Estaldurarekin (*Recall*) gertatzen den moduan, egia positiboen tasa (*TPR*) eta faltsu positiboen tasa
(*FPR*) ezin dira errendimendu-metrika isolatu gisa erabili:

* **Beti `1` iragartzen duen eredua**. Sekula `0` iragarriko ez duenez, $FN=0$ eta $TP>0$, beraz $\text{TPR}=1$
* **Beti `0` iragartzen duen eredua**. Sekula `1` iragarriko ez duenez, $FP=0$ eta $TN>0$, beraz $\text{FPR}=0$


### Hiru ereduen errendimendu metrikak

*Scikit-learn*-ek errendimendu metrika guzti hauek definituak ditu `sklearn.metrics` moduluan:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

Auzazko eredua:

In [ ]:
np.random.seed(42)
y_train_predict = np.random.randint(0, 2, size=y_train.size)
y_test_predict = np.random.randint(0, 2, size=y_test.size)
for score in [accuracy_score, precision_score, recall_score, f1_score] :
    score_train = score(y_train, y_train_predict)
    score_test = score(y_test, y_test_predict)
    print(f"{score.__name__} - train: {score_train:.3f}  test: {score_test:.3f}")

Moda eredua:

In [ ]:
y_mode = y_train.mode()[0]
y_train_predict = np.full(y_train.shape, y_mode)
y_test_predict = np.full(y_test.shape, y_mode)
for score in [accuracy_score, precision_score, recall_score, f1_score] :
    score_train = score(y_train, y_train_predict)
    score_test = score(y_test, y_test_predict)
    print(f"{score.__name__} - train: {score_train:.3f}  test: {score_test:.3f}")

ACME sailkatzailea:

In [ ]:
model = grid_search.best_estimator_
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
for score in [accuracy_score, precision_score, recall_score, f1_score] :
    score_train = score(y_train, y_train_predict)
    score_test = score(y_test, y_test_predict)
    print(f"{score.__name__} - train: {score_train:.3f}  test: {score_test:.3f}")

`train` azpimultzoko emaitzak, lau metrika ezberdinetan:

| System | $\mathbf{Accuracy}$ | $\mathbf{Precision}$ | $\mathbf{Recall}$ | $\mathbf{F1-score}$ |
| :--- | :--- | :--- | :--- | :--- |
| **Baseline-rand** | 0.497 | 0.621 | 0.510 | 0.560 |
| **Baseline-mode** | 0.629 | 0.629 | **1.000** | 0.772 |
| **ACMEClassifier** | **0.987** | **0.986** | 0.993 | **0.990** |

`test` azpimultzoko emaitzak, lau metrika ezberdinetan:

| System | $\mathbf{Accuracy}$ | $\mathbf{Precision}$ | $\mathbf{Recall}$ | $\mathbf{F1-score}$ |
| :--- | :--- | :--- | :--- | :--- |
| **Baseline-rand** | 0.509 | 0.627 | 0.521 | 0.569 |
| **Baseline-mode** | 0.623 | 0.623 | **1.000** | 0.768 |
| **ACMEClassifier** | **0.982** | **0.973** | **1.000** | **0.986** |

### Hausnarketa

Erabiliko dugun errendimendu-metrika **aurrez ezarri behar dugu**, ez du inolako zentzurik lanaren helburua amaieran diseinatzea.

## 10 - Probabilitateen iragarpena, erabaki-atalasa eta erabaki-kurbak

Eredu sailkatzaileak, orohar, aldagai iragarle adibide bat emanda, adibide hori klase bakoitzari esleitzeko probabilitatea kalkulatzeko gai dira. `model.predict_proba(X)` funtzioak, `(N,K)` tamainako array bat bueltatuko du, `N` adibide kopurua delarik (`X` datu-matrizearen ilara kopurua) eta `K` klase kopurua (gure kasuan, 2):

In [ ]:
proba = model.predict_proba(X_train)
proba[:10,:]

Adibide bakoitzarentzat klase guztien probabilitateen batura 1 izan behar denez, ilara bakoitzaren batura ere 1 izango da:

In [ ]:
proba[:10].sum(axis=1)

Sailkatzaileei predikzioak egiteko eskatzen diegunean, probabilitate handiena duen klasea zein ote den konprobatzen dute:

In [ ]:
print(proba[:10].argmax(axis=1))
print(model.predict(X_train[:10]))

`X_train` osoarekin konproba dezakegu, ea biak baliokideak diren:

In [ ]:
#proba.argmax(axis=1) == model.predict(X_train)
(proba.argmax(axis=1) == model.predict(X_train)).all()

Bi klase ditugunean, klase batek probabilitate handiena izateko, haren balioa $0.5$ baina handiagoa izan beharko da:

In [ ]:
print((proba[:,1] > 0.5)[:10])
print((proba[:,1] > 0.5).astype('int')[:10])

Berriro ere, `train` guztiarekin froga genezake:

In [ ]:
predict_train = (proba[:,1] > 0.5).astype('int')
(predict_train == model.predict(X_train)).all()


$\tau = 0.5$ atalase-balio (*threshold*) bat da. Eredua finko utzirik atalasea aldatzen badugu, iragarpenak aldatuko dira:

* $\tau < 0.5$ : Ereduak *erraztasun* handiagoa izango du `1` klasea (positiboak) iragartzeko (`0` iragarpen kopurua txikitu)
   * Positiboak &uarr; eta beraz TP &uarr; FP &uarr;
   * Negatiboak &darr; eta beraz TN &darr; FN &darr;
   * Doitasuna &darr; (*Precision* &darr;)
   * Estaldura &uarr; (*Recall* &uarr;)
* $\tau > 0.5$ : Ereduak *zailtasun* handiagoa izango du `1` klasea (positiboak) iragartzeko (`0` iragarpen kopurua handitu)
   * Positiboak &darr; eta beraz TP &darr; FP &darr;
   * Negatiboak &uarr; eta beraz TN &uarr; FN &uarr;
   * Doitasuna &uarr; (*Precision* &uarr;)
   * Estaldura &darr; (*Recall* &darr;)



### Doitasuna-Estaldura erabaki-kurba

$\tau \in [0,1]$ atalase balio bakoitzak Precision-Recall erlazio bat izango du. Erlazio honi **Doitasuna-Estaldura (Precision-Recall) truke-harremana (*Trade-off*)** edo **Doitasuna-Estaldura erabaki-kurba** deritzo:

<img src="img/precision-recall.png" style="display: block; margin: 0 auto; width: 70%; height: auto;" alt="Precision/Recall Trade-off curve"/>

Doitasuna-Estaldura truke-harremanaren grafika *Sciki-learn*-eko `PrecisionRecallDisplay.from_predictions` funtzioarekin lortu daiteke. Funtzio honek benetako klaseen `y` bektorea eta klase positiboen probabilitate bektorea behar ditu:

```python
# positive class probabilities, for example:
probpos = model.predict_proba(X)[:, 1]
PrecisionRecallDisplay.from_predictions(y, probpos, name="My Classifier")
```

Sistema bat baina gehiago ere grafika berean jar ditzakegu:
```python
probpos1 = ...
probpos2 = ...
probpos3 = ...
fig, ax = plt.subplots(figsize=(5, 5))
PrecisionRecallDisplay.from_predictions(y, probpos1, name="My Classifier 1", ax=ax)
PrecisionRecallDisplay.from_predictions(y, probpos2, name="My Classifier 2", ax=ax)
PrecisionRecallDisplay.from_predictions(y, probpos3, name="My Classifier 3", ax=ax)
```

**Baseline-rand** erreferentzia-ereduaren klase positiboaren probabilitate bektorea: 

In [ ]:
np.random.seed(42)
probpos_train_rand = np.random.random(y_train.shape)
probpos_train_rand[:5]

**Baseline-mode** erreferentzia-ereduaren klase positiboaren probabilitate bektorea: 

In [ ]:
probpos_train_mode = np.full(y_train.shape,1.0)
probpos_train_mode[:5]

**ACMEClassifier** ereduaren klase positiboaren probabilitate bektorea: 

In [ ]:
probpos_train_acme = model.predict_proba(X_train)[:, 1]
probpos_train_acme[:5]

Hirurak batera irudikatu:

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

fig, ax = plt.subplots(figsize=(5, 5))
PrecisionRecallDisplay.from_predictions(y_train, probpos_train_rand, name="Baseline-rand", ax=ax)
PrecisionRecallDisplay.from_predictions(y_train, probpos_train_mode, name="baseline-mode", ax=ax)
PrecisionRecallDisplay.from_predictions(y_train, probpos_train_acme, name="ACMEClassifier", ax=ax);

Irudiak erakusten duen *AP* balioa batazbesteko doitasuna (*Average Precision*) da, eredu sailkatzaile baten ezaugarri agregatu bat da, errendimendu metrika gisa erabil daitekeena.

### ROC (*Receiver Operating Characteristic*) kurba

ROC (*Receiver Operating Characteristic*) kurbak $\tau \in [0,1]$ atalase balioa aldatzean gertatzen den Egiazko Positiboen Tasa (*TPR*) eta Positibo Faltsuen Tasaren (*FPR*) arteko erlazioa adierazten du:

<img src="img/roc.png" style="display: block; margin: 0 auto; width: 70%; height: auto;" alt="Receiver Operating Characteristic curve"/>


ROC kurbaren azpiko azalera (**AUC**, *Area Underthe Curbe) eredu sailkatzaile baten ezaugarri agregatu bat da, errendimendu metrika gisa erabil daitekeena. Gainera, ROC kurbaren kasuan auzazko sailkatzaile batek lerro zuzen bat marraztuko du:

<img src="img/roc2.png" style="display: block; margin: 0 auto; width: 60%; height: auto;" alt="Receiver Operating Characteristic curve"/>


ROC curbaren grafika *Sciki-learn*-eko `RocCurveDisplay.from_predictions` funtzioarekin lortu daiteke. Funtzio honek benetako klaseen `y` bektorea eta klase positiboen probabilitate bektorea behar ditu:

```python
# positive class probabilities, for example:
probpos = model.predict_proba(X)[:, 1]
RocCurveDisplay.from_predictions(y, probpos, name="My Classifier")
```

Sistema bat baina gehiago ere grafika berean jar ditzakegu:
```python
probpos1 = ...
probpos2 = ...
probpos3 = ...
fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_predictions(y, probpos1, name="My Classifier 1", ax=ax)
RocCurveDisplay.from_predictions(y, probpos2, name="My Classifier 2", ax=ax)
RocCurveDisplay.from_predictions(y, probpos3, name="My Classifier 3", ax=ax)
```

Aurretik lortutako klase positiboaren probabilitate bektoreak erabiliz: 

In [ ]:
from sklearn.metrics import RocCurveDisplay

#np.random.seed(42)
#probpos_train_rand = np.random.random(y_train.shape)

fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_predictions(y_train, probpos_train_rand, name="Baseline-rand", ax=ax)
RocCurveDisplay.from_predictions(y_train, probpos_train_mode, name="baseline-mode", ax=ax)
RocCurveDisplay.from_predictions(y_train, probpos_train_acme, name="ACMEClassifier", ax=ax);

## 11 - Errendimendu-metrikak vs Galera-funzioa (*Loss*)

Erregresio problemetan, errendimendu-metrikak eta galera-funtzioa, kontzeptualki gauza bera izan daitezke:
* $P \propto -MSE \propto -RMSE$
* $\mathcal{L} \propto MSE \propto RMSE$

Sailkapen problema batean, ordea, orain arte aztertutako errendimendu-metrikak (nahasmen matrizetik eratorritakoak) ezin dira galera funtzio gisa erabili:
* $P = \text{Accuracy}, \text{F1-score}, \dots, f(\text{conf\_matrix})$
* $\mathcal{L} \ne \text{Accuracy}, \text{F1-score}, \dots, f(\text{conf\_matrix})$

Zergatik?

Eredu gehienen doiketan **gradianteak** erabiltzen dira. Nahasmen matrizetik eratorritako funtzio guztiak **mailakatuak** (ez-jarraituak) dira, <u>hiperparametroen aldaketa txiki batek ez du metrikaren aldaketarik suposatzen</u>. Gradianteak nuluak direnez, ezinezkoa da gradianteetan oinarritutako doitze metodorik erabiltzea.

Irtenbidea: <u>Jarraia den galera funtzio bat erabiltzea</u>

### Entropia Gurutzatua (*Log Loss, Binary Cross-Entropy*)

Ereduen erabakietan oinarritu beharrean, Entropia Gurutzatua erabaki horiek hartzeko ereduak sortutako/erabilitako probabilitateetan oinarritzen da:

$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

non $\hat{p}_i$ eta $y_i$ ereduak $i$. laginari esleitutako klase positiboaren probabilitatea eta lagin horri dagokion benetako klasea ($y_i \in {0,1}$) diren.

Entropia Gurutzatuak ondoko ezaugarriak ditu:
* Funtzio leun (jarraitu) bat denez, gradiante metodoak erabil daitezke.
* Gogor zigortzen ditu konfiantza handiz egindako akatsak (adibidez, $y=1$ denean $\hat{p}=0.01$ iragartzea)
* Ereduak laginei buruz azaldu ezin dezakeen **informazioa** adierazten du. Logaritmoaren oinarria $2$ denean, **bit**-etan adierazten du.
   * Eredua erreferentzitzat hartuta, lagin bakoitza azaltzeko behar den batazbesteko informazioa 

Azter ditzagun ondoko ereduen entropia gurutzatua (2 oinarria erabiliz):

1. Ezer ez dakien eredua (*Non-Informative*): $\hat{p}_i = 0.5$
1. Auzazko eredua: $\hat{p}_i =  r_i \in [0,1]_{random}$
1. Beti `1` eredua: $\hat{p}_i = 1$
1. Probabilitate konstanteko eredua: $\hat{p}_i = p_{kte}$
1. ACMEClasiffier eredua

#### 1. Ezer ez dakien eredua (*Non-Informative*): $\hat{p}_i = 0.5$
$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log_2(\hat{p}_i) + (1 - y_i) \log_2(1 - \hat{p}_i) \right]$$

$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^{N} \log_2(0.5) = -\frac{1}{N} \sum_{i=1}^{N} (-1) = 1 \, \text{bit}$$

Ereduak ez du inolako informaziorik eskaintzen. Bi klase posible ditugunez, lagin bakoitzeko bit bat behar dugu laginaren informazio guztia adierazteko (ereduak laginei buruz azaldu ezin dezakeen informazio kantitate maximoa). Edozein ereduren entropia gurutzatuaren maximo onargarria `1 bit` izango da (informazio ezaren muga).

$$0 < \mathcal{L}(y, \hat{p}) < 1$$

#### 2. Auzazko eredua: $\hat{p}_i = r_i \in [0,1]_{random}$
$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log_2(\hat{p}_i) + (1 - y_i) \log_2(1 - \hat{p}_i) \right]$$

$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^{N} \log_2(r_i) = ??? $$

Enpirikoki kalkula genezake:

In [ ]:
for N in [1_000, 10_000, 100_000, 1_000_000, 10_000_000] :
    r = np.random.rand(N)
    loss_nats = -np.log(r).mean()       # logaritmo nepertarra --> nat
    loss_bits = loss_nats/np.log(2.0)  # 2 oinarria --> bit
    print(f'{N=}  {loss_nats=:0.2f} {loss_bits=:0.2f}')

Ereduak lagin bakoitzeko azaldu ezin dezakeen informazioa $1.44$ bit dira, baina lagin bakoitzak gehiebez bit bakarreko informazioa darama, nola da posible? Ereduak informazioa eskaini beharrean **kendu** egiten al digu?

Nolabait, bai. Ereduak ez daki ezertxo ere laginei buruz, eta gainera probabilitateak asmatzen (haluzinatzen) ditu. Asmatutako balio hauek batzuetan benetako klasera hurbiltzen dira (zigor **txikia**) baino besteetan urrundu (zigor **oso handia**), eta batazbesteko zigorra informazio ezarena baina handiagoa da. Batazbesteko errore extra horrek **asmatutako** informazioa adierazten du.

#### 3. Beti `1` eredua: $\hat{p}_i = 1$
$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log_2(\hat{p}_i) + (1 - y_i) \log_2(1 - \hat{p}_i) \right]$$

$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \left( \sum_{y_i=1} \log_2(1) + \sum_{y_i=0} \log_2(0) \right) = -\infty$$

`1` laginak guztiz ongi asmatu dira ($\hat{p}_i=1$) eta lagin hauek ez dute zigorrik jasotzen. `0` laginetan ordea, $1-\hat{p}_i=0$ probabilitatea esleitzean, zigorra infinitoa da. Sekulako gezurrak esateko gai da eredua. Informazioaren ikuspuntutik, gerta ezin daitekeen zerbait azaltzeko behar den bit kopurua **infinitoa** da.

$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$


#### 4. Probabilitate konstanteko eredua: $\hat{p}_i = p_{kte}$

$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log_2(\hat{p}_i) + (1 - y_i) \log_2(1 - \hat{p}_i) \right]$$

$$\mathcal{L}(y, \hat{p}) = -\frac{1}{N} \left( \sum_{y_i=1} \log_2(p) + \sum_{y_i=0} \log_2(1-p) \right) = -\frac{N_1 \log_2(p) + N_0 \log_2(1-p)}{N}$$

Hemendik 1. eta 3. ereduen galerak lor daitezke, edo baita ere "Beti `0` eredua ( $\hat{p}_i = 1$)":

$$\mathcal{L}(y, \hat{p}) = -\infty$$

Orohar, $p=\frac{N_1}{N}$ denean (ereduaren probabilitatea benetako probabilitatearekin bat datorrenean), entropia gurutzatuak balio minimoa izango du:

<img src="img/entropia-gurutzaptua-p-kte.jpg" style="display: block; margin: 0 auto; width: 60%; height: auto;" alt="Entropia-gurutzaptua-p-kte"/>

Irudi hau (antzekoa) ere eskuz lor dezakegu:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Datuen parametroak (N_1/N ratioa aldatu dezakezu nahi izanez gero)
N  = 1000
N_1 = 350
r = N_1 / N  #  (klase positiboaren proportzioa)
p = np.linspace(0.001, 0.999, 500)

# Entropia gurutzatuaren kalkulua (bit-etan)
loss = -(r * np.log2(p) + (1 - r) * np.log2(1 - p))
min_loss = -(r * np.log2(r) + (1 - r) * np.log2(1 - r))

# 2. Irudia sortu
fig, ax = plt.subplots(figsize=(8, 4.8), dpi=120)

# Kurba nagusia
ax.plot(p, loss, color='#1e3a8a', linewidth=2.5, 
        label=r'$\mathcal{L}(y, \hat{p})$')

# 1.0 bit-eko erreferentzia-muga (non-informative)
ax.axhline(1.0, color='#ea580c', linestyle='--', linewidth=1.8, 
           label='1.0 bit muga (non-informative)')

# Puntu garrantzitsuak: Minimoa (p = N1/N) eta p = 0.5
ax.axvline(r, color='#94a3b8', linestyle=':', linewidth=1.2)
ax.scatter([r], [min_loss], color='#dc2626', s=60, zorder=5, 
           label=f'Minimoa ($p = N_1/N = {r}$)')

ax.axvline(0.5, color='#94a3b8', linestyle=':', linewidth=1.2)
ax.scatter([0.5], [1.0], color='#2563eb', s=60, zorder=5, 
           label='Ezer ez dakien eredua ($p = 0.5$)')

# 3. Ardatzen mugak eta etiketak
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.8)
ax.set_xlabel('$p$ (Klase positiboaren probabilitatea)', fontsize=11)
ax.set_ylabel(r'Entropia gurutzatua $\mathcal{L}(y, p)$ [bit]', fontsize=11)
ax.set_title(f'Entropia gurutzatua $p$ konstantearen arabera ($N_1/N = {r}$)', fontsize=12)

# Estiloa garbitu
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(frameon=True, fontsize=10, loc='upper right')

plt.tight_layout()
plt.show()

### Hausnarketa

Entropia gurutzatua horren eraginkorra bada, informazio unitateetan neurtu badaiteke eta sailkapen ereduak bera minimizatzen doitzen badira, zergatik ez da errendimendu metrika gisa erabiltzen, nahasmen matrizetik eratorritako metrikak baztertuz?

1. Batzuetan hala egiten da
1. **Benetako erabilpena.** Mundu errealak erabaki diskretuak (ekintzak) ezkatzen ditu, ez probabilitateak (informazioa). Erabakiak hartzeko atalase balioak erabiliko dira eta entropia gurutzatuak ez digu erabaki horien kalitatearen berri ematen zuzenean.
1. **Interpretabilitatea eta negozio-hizkuntza**. Informazioaren teorian bit-ak edo nat-ak oso dotoreak diren arren, ez dira intuitiboak erabiltzaile edo bezero batentzat.
1. **Muturreko akatsekiko sentikortasun gehiegizkoa.** Logaritmoaren izaera dela eta ($\lim_{p \to 0} -\log_2(p) = \infty$), entropia gurutzatuak izugarri zigortzen ditu konfiantza handiz egindako akatsak (gezurra galantak!).
1. **Probabilitateen kalibrazioa vs. Bereizmen-gaitasuna.** Eredu batek klaseak primeran ordenatu eta bereiz ditzake (adibidez, negatibo guztiei $\hat{p}=0.45$ esleituz eta positibo guztiei $\hat{p}=0.55$).
   * $\tau = 0.5$ atalasearekin, eredu horrek % 100eko Accuracy, Precision eta Recall izango ditu (erabaki perfektuak hartzen ditu!).
   * Baina bere entropia gurutzatua nahiko kaskarra izango da ($\approx 0.86\text{ bit}$), probabilitateak $0.5$-etik oso gertu daudelako (ereduak "zalantza" handia duelako, gaizki kalibratuta dagoelako).

Laburbilduz, zergatik behar ditugu biak?

* **Galera-funtzioa (Entropia gurutzatua)** ereduaren iparrorratza da entrenamenduan: pauso leun eta deribagarriak emanez probabilitate onenak ikasten laguntzen dio.
* **Errendimendu-metrikak (Accuracy, , F1-score...)** ereduaren azterketa praktikoa dira: behin ikaskuntza amaituta, eredu horrek mundu errealean hartuko dituen erabaki zehatzen onurak eta kalteak gizakiontzat ulergarria den hizkuntzan neurtzen dituzte.